# 01 - Data Loading and Exploratory Data Analysis
## Predicting Human Intestinal Absorption (HIA) Using ML and Explainable AI (SHAP)
**Student:** Mohammad Karamath Fardeen | ID: 25251265  
**Programme:** MSc Data Science & Analytics, Maynooth University  
**Supervisor:** Kolawole Adebayo  
**Year:** 2025-2026

## 1. Install and Import Libraries

In [ ]:
# Install required libraries if not already installed
# !pip install PyTDC rdkit scikit-learn xgboost lightgbm shap matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
SEED = 42
print('Libraries imported successfully!')

## 2. Load HIA_Hou Dataset from TDC

In [ ]:
from tdc.single_pred import ADME

# Load dataset
data = ADME(name='HIA_Hou')
split = data.get_split(method='scaffold', seed=SEED)

train_df = split['train']
valid_df = split['valid']
test_df  = split['test']

# Save raw splits
os.makedirs('data/raw', exist_ok=True)
train_df.to_csv('data/raw/hia_train.csv', index=False)
valid_df.to_csv('data/raw/hia_valid.csv', index=False)
test_df.to_csv('data/raw/hia_test.csv', index=False)

print(f'Train: {len(train_df)} | Valid: {len(valid_df)} | Test: {len(test_df)}')
print(f'Total: {len(train_df)+len(valid_df)+len(test_df)} molecules')
train_df.head()

## 3. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, df) in zip(axes, [('Train', train_df), ('Valid', valid_df), ('Test', test_df)]):
    counts = df['Y'].value_counts().sort_index()
    bars = ax.bar(['Low (0)', 'High (1)'], counts.values, color=['#E74C3C', '#2ECC71'])
    ax.set_title(f'{name} set (n={len(df)})')
    ax.set_ylabel('Count')
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(count), ha='center', fontweight='bold')

plt.suptitle('HIA Class Distribution Across Splits', fontsize=13, fontweight='bold')
plt.tight_layout()
os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nClass imbalance in training set:')
print(train_df['Y'].value_counts())

## 4. Compute RDKit Molecular Descriptors

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from tqdm import tqdm

DESCRIPTOR_FUNCTIONS = {
    'MolWt':         Descriptors.MolWt,
    'LogP':          Descriptors.MolLogP,
    'TPSA':          Descriptors.TPSA,
    'HBD':           rdMolDescriptors.CalcNumHBD,
    'HBA':           rdMolDescriptors.CalcNumHBA,
    'RotBonds':      rdMolDescriptors.CalcNumRotatableBonds,
    'RingCount':     rdMolDescriptors.CalcNumRings,
    'AromaticRings': rdMolDescriptors.CalcNumAromaticRings,
    'HeavyAtoms':    Descriptors.HeavyAtomCount,
    'FractionCSP3':  rdMolDescriptors.CalcFractionCSP3,
    'MolMR':         Descriptors.MolMR,
    'NumHeteroatoms':rdMolDescriptors.CalcNumHeteroatoms,
}

def smiles_to_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {name: fn(mol) for name, fn in DESCRIPTOR_FUNCTIONS.items()}

def compute_features(df):
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        feats = smiles_to_descriptors(row['Drug'])
        if feats:
            feats['Drug_ID'] = row['Drug_ID']
            feats['SMILES']  = row['Drug']
            feats['Y']       = int(row['Y'])
            records.append(feats)
    return pd.DataFrame(records)

os.makedirs('data/processed', exist_ok=True)
print('Computing descriptors...')
for split_name, df in [('train', train_df), ('valid', valid_df), ('test', test_df)]:
    feat_df = compute_features(df)
    feat_df.to_csv(f'data/processed/hia_{split_name}_features.csv', index=False)
    print(f'{split_name}: {len(feat_df)} molecules, {len(feat_df.columns)} columns')

## 5. Feature Distribution by Class

In [ ]:
train_feat = pd.read_csv('data/processed/hia_train_features.csv')
FEATURE_COLS = list(DESCRIPTOR_FUNCTIONS.keys())

key_features = ['TPSA', 'LogP', 'MolWt', 'HBD', 'HBA', 'RotBonds']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, feat in zip(axes, key_features):
    sns.kdeplot(data=train_feat, x=feat, hue='Y', fill=True, alpha=0.4, ax=ax,
                palette={0: '#E74C3C', 1: '#2ECC71'})
    ax.set_title(feat)
    ax.legend(title='HIA', labels=['High (1)', 'Low (0)'])

plt.suptitle('Molecular Descriptor Distributions by HIA Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
corr = train_feat[FEATURE_COLS].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Matrix — Molecular Descriptors', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()